# Chapter 7: Search In-Depth

## Topic 1: Sparse Retrieval — TF-IDF → BM25

---

### 1. Concept and Intuition

**What your project already does — Chapter 1 baseline:**

- `classify_by_keyword_rules()` checks whether any word from `fd_keyword_groups.txt` appears in an email
- Four keyword groups cover the FD domain: maturity, machurity, byaj (maturity/payout group) — premature, withdrawal, nikalna, naya fd (premature/rollover group) — fd a/c, fd reference, jama kiya (account reference group) — nominee, interest rate, tenure (nominee/tenure group)
- `fd_negation_phrases.txt` provides 16 veto terms: loan, emi, insurance, card, login, otp, app, kyc, branch, statement and more
- Decision rule: FD terms present + negation absent → FD | negation present + FD absent → Non-FD | both present → Multiple Category
- This is the crudest possible sparse retrieval: binary word presence or absence, no frequency weighting, no corpus statistics, no document length awareness

**Why "sparse":**

- Representing a document as a vector means one dimension per word in the entire vocabulary
- Your 2,500-email corpus has thousands of unique words
- Any single email averages ~31 words (FD class) — so only ~31 of thousands of dimensions are non-zero
- The vector is almost entirely zeros — hence sparse
- Dense embeddings (Chapter 3) have all 384 dimensions non-zero for every document — a fundamentally different approach solving a different failure mode

**What problem BM25 solves that your rule engine cannot:**

- Your EDA found 145 Non-FD emails containing FD-related words — the rule engine mislabels all of these because it cannot ask "is this word rare enough in the corpus to actually be informative?"
- It cannot ask "does appearing 10 times make this email 10x more relevant, or just slightly more?"
- For RAG retrieval specifically: a query like "what is the penalty for premature FD closure?" needs ranked results from 17 policy pages, not a binary yes/no — BM25 provides mathematically correct ranking, the rule engine cannot rank at all

---

### 2. TF-IDF — What It Gets Right and Where It Breaks

**The formula:**

```text
TF(t, d)    = count of term t in document d / total terms in d
IDF(t)      = log( N / df(t) )
TF-IDF(t,d) = TF(t,d) × IDF(t)

N     = total documents in corpus
df(t) = number of documents containing term t
```

**Real numbers from your project (Chapter 0, `02_Text_Representation.ipynb`):**

From the first email "Branch gaya tha... Two wheeler loan ki EMI is mahine zyada..." (label: Non-FD):

- "gaya" → appears 2 times, in 613 of 2,500 emails → TF-IDF = 0.750 (highest, because of repetition)
- "ho" → appears 1 time, in 598 of 2,500 emails → TF-IDF = 0.379 (rarest word → boosted)
- "loan" → appears 1 time, in 642 of 2,500 emails → TF-IDF = 0.368 (topically meaningful)
- "is" → appears 1 time, in 1,053 of 2,500 emails → TF-IDF = 0.291 (common → downweighted)
- "hai" → appears 1 time, in 1,190 of 2,500 emails → TF-IDF = 0.272 (most common → lowest weight)

This already demonstrates TF-IDF working correctly: common filler words like "hai" and "is" are suppressed; topically meaningful "loan" gets decent weight; rare "ho" gets boosted.

**What TF-IDF gets right:**

- Common words like "hai", "sir", "the" appear in most documents → IDF near zero → suppressed automatically without a stop-word list
- Rare meaningful terms like "machurity", "byaj", "premature withdrawal" appear in few documents → high IDF → strong discriminating power
- Exact match is guaranteed — a query containing "BJ2019FD7717" will surface every document where that exact string appears (dense embeddings may fail on novel tokens the model has not seen)
- Cheap: no GPU, no model to load, millisecond search, zero per-query API cost

**Where TF-IDF breaks — Problem 1: Term Frequency Saturation:**

- TF grows linearly — a document mentioning "maturity" 40 times gets 40× the TF score of a document mentioning it once
- But mentioning "maturity" 40 times across a long SOP does not make it 40× more relevant than a concise FAQ answer mentioning it once
- Your `03_FD_SOPs.pdf` likely repeats terms across multi-step procedures — TF-IDF ranks it above `01_FD_FAQ.pdf` purely due to repetition, not relevance
- This is the linear relationship problem: raw frequency ≠ relevance

**Where TF-IDF breaks — Problem 2: No Document Length Normalisation:**

- A 10-page policy document and a 1-paragraph FAQ both contain "interest"
- The policy document has more total words, so TF divides by a larger denominator — but this doesn't fully correct for the fact that long documents simply contain more of every word
- Without proper length normalisation, long documents systematically beat short ones for identical vocabulary — not because they are more relevant but because they are longer
- Your corpus has exactly this mix: short FAQ Q&A pairs vs. long dense policy clauses vs. multi-step SOPs

---

### 3. BM25 — The Two Fixes That Made It the Industry Default

**Background:**
BM25 stands for Best Match 25 — the 25th iteration of the Okapi probabilistic retrieval model from City University London, developed through the 1980s–1990s. It is still the default sparse retrieval algorithm in Elasticsearch, OpenSearch, Solr, and most production search systems.

**The formula:**

```text
BM25(d, q) = Σ  IDF(t) × [ TF(t,d) × (k1 + 1) ]
             t∈q          [ TF(t,d) + k1 × (1 - b + b × |d|/avgdl) ]

IDF(t)  = log( (N - df(t) + 0.5) / (df(t) + 0.5) + 1 )

t      = each query term
d      = document being scored
|d|    = length of document d in words
avgdl  = average document length across the corpus
k1     = term frequency saturation parameter (default: 1.2)
b      = length normalisation parameter (default: 0.75)
```

**Fix 1 — Term Frequency Saturation via k1:**

- The numerator `TF(t,d) × (k1 + 1)` grows with TF
- The denominator `TF(t,d) + k1` also grows with TF
- As TF → ∞, the fraction asymptotically approaches `(k1 + 1)` — it never exceeds this ceiling
- With k1 = 1.2: TF of 1 gives ~1.82, TF of 40 gives ~2.17 — the verbose document gets only 19% more credit for 40× the repetition
- Compare TF-IDF: 40× repetition gives 40× the score
- For your corpus: the SOP document repeating "maturity" across 8 steps now gets almost the same score as the FAQ mentioning it once — relevance gap closed

**k1 behaviour:**
- k1 = 0 → binary (word present or absent, frequency irrelevant)
- k1 = 1.2 → fast saturation (default, works well for most corpora)
- k1 = 2.0 → slower saturation (gives more credit to repetition before ceiling)

**Fix 2 — Document Length Normalisation via b:**

- The denominator term `(1 - b + b × |d|/avgdl)` adjusts scoring relative to average document length
- When `|d| = avgdl`: term equals 1, no adjustment applied
- When `|d| > avgdl` (longer document): denominator increases → score decreases → penalises verbosity
- When `|d| < avgdl` (shorter document): denominator decreases → score increases → rewards conciseness
- For your corpus: policy chunks (long) no longer automatically beat FAQ entries (short) for the same query

**b behaviour:**
- b = 0 → no length normalisation at all (degenerates toward raw TF behaviour)
- b = 0.75 → partial normalisation (default, empirically best across many IR benchmarks)
- b = 1.0 → full length normalisation

**The improved IDF in BM25 vs TF-IDF:**

```text
BM25 IDF:  log( (N - df(t) + 0.5) / (df(t) + 0.5) + 1 )
TF-IDF IDF: log( N / df(t) )
```

- The `+ 0.5` smoothing prevents log(0) when a term appears in every document
- Keeps IDF positive even for very common terms (plain TF-IDF can produce negative IDF in some variants)
- More numerically stable across corpus size changes

---

### 4. How It Is Implemented in This Project

**The rule engine is binary sparse retrieval — BM25 is ranked sparse retrieval:**

```python
# Chapter 1 classify_by_keyword_rules() — binary presence/absence
# fd_keyword_groups.txt has 4 groups, fd_negation_phrases.txt has 16 veto terms

def classify_by_keyword_rules(subject: str, content: str) -> str:
    text = (subject + " " + content).lower()
    has_fd_term = any(kw in text for group in FD_KEYWORD_GROUPS for kw in group)
    has_negation = any(neg in text for neg in NEGATION_PHRASES)
    if has_fd_term and not has_negation:
        return "FD"
    elif has_negation and not has_fd_term:
        return "Non-FD"
    elif has_fd_term and has_negation:
        return "Multiple Category"
    else:
        return "ambiguous"
```

This treats every keyword as equally important regardless of how rare or common it is across your 2,500-email corpus, and regardless of how many times it appears in this specific email.

**BM25 for RAG retrieval (using rank_bm25):**

```python
from rank_bm25 import BM25Okapi

# Build the index: tokenize each document chunk
tokenized_corpus = [doc["page_content"].lower().split() for doc in all_docs]
bm25 = BM25Okapi(tokenized_corpus, k1=1.2, b=0.75)

# Query: same tokenization as index
query = "premature withdrawal penalty FD"
query_tokens = query.lower().split()
scores = bm25.get_scores(query_tokens)

# scores[i] = BM25 score for chunk i — higher means more relevant
top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]
for idx in top_indices:
    print(f"Score {scores[idx]:.3f}: {all_docs[idx]['page_content'][:80]}")
```

**Connecting to the RAG pipeline (your 17 pages across 4 documents):**

- At ingest time: tokenize every chunk from `01_FD_FAQ.pdf`, `02_FD_Product_Guide.pdf`, `03_FD_SOPs.pdf`, `04_FD_Policy_Reference.pdf` and build a BM25 index
- At query time: tokenize the customer's query and call `bm25.get_scores()`
- Return top-k chunks by BM25 score to the generation layer
- This replaces the brute-force "does any keyword match?" logic with a proper ranked retrieval

**The connection Topic 5 of this chapter will make explicit:**

- `classify_by_keyword_rules()` is performing the sparse retrieval step of what would be hybrid search
- It checks "is any FD-domain word present?" — the sparse retrieval signal
- BM25 formalises this as "which documents are most relevant given this query according to term statistics?" — ranked, not binary
- Topic 5 builds them side by side so the evolution is visible

---

### 5. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

**Vocabulary mismatch — the one thing BM25 cannot fix:**

- A customer writes "mera paisa fasa hua hai" — none of those words appear in your policy chunks, which use English terms like "premature closure" and "maturity proceeds"
- BM25 score is exactly zero — not low, exactly zero
- This is the primary motivation for dense retrieval (Topic 2) and hybrid search (Topic 4)
- Your 64.4% Hinglish corpus makes this failure mode frequent, not rare

**Hinglish-specific BM25 problems:**

- "Machurity" and "maturity" are completely different tokens to BM25 — it has no phonetic or semantic similarity awareness
- "Byaj" and "interest" are different tokens — BM25 cannot know they mean the same thing unless both appear in the same document
- `fd_keyword_groups.txt` manually solved this for classification by grouping both spellings: "maturity, machurity" and "interest, byaj" in the same group
- For BM25 retrieval over policy chunks, you face the same problem — the query uses "byaj" but the policy chunk uses "interest" → BM25 score = 0
- **Fix options:** query expansion before BM25 (replace "byaj" with "byaj OR interest"), a normalisation layer mapping Hinglish variants to canonical English before indexing, or hybrid search with dense retrieval to catch what BM25 misses

**IDF instability with a small corpus:**

- Your RAG knowledge base has 17 pages — when "maturity" appears in 10 of 17 pages, its IDF is low
- IDF computed on 17 pages is completely different from IDF computed on 10,000 pages of the same domain
- Never threshold on absolute BM25 scores (e.g., "relevant if score > 5") — the scale shifts with corpus size
- Always rank relative to other results within the same query's output
- Always re-evaluate retrieval after large corpus expansions

**Scaling — why `rank_bm25` has a limit:**

- `rank_bm25` does not build a true inverted index — it scans all documents for each query
- At 17 pages this is invisible — at 100,000 chunks it becomes slow (O(n) per query)
- Production systems (Elasticsearch, OpenSearch) maintain an inverted index: a data structure mapping each vocabulary term to a sorted list of (document_id, term_frequency) pairs
- A query with 3 terms fetches 3 posting lists and merges — only documents containing at least one query term are visited
- This reduces per-query work from O(n) to O(k) where k << n

**Latency (concrete numbers):**

- BM25 over your 17-page RAG corpus: under 1 millisecond
- BM25 over 500,000 documents with a proper inverted index: 5–20 milliseconds
- Dense retrieval (embedding call + vector search): 50–200 milliseconds
- BM25 is dramatically faster — the cheapest retrieval method available

**Cost:**

- BM25 has zero per-query API cost — no model calls, no GPU
- Compare to: every dense retrieval query requires an embedding model call; every reranked query (Topic 7) requires a cross-encoder call per candidate
- BM25 is your cheapest signal — use it first, always

**Monitoring — what to track:**

- Queries producing zero BM25 scores across all corpus documents — these are vocabulary gap signals, each is a potential query expansion or normalisation opportunity
- Average BM25 score of top-1 result over time — downward drift signals corpus staleness or query distribution shift (users asking about new products not yet in the corpus)
- Fraction of queries where BM25 top-5 and dense top-5 have zero overlap — high overlap means they are largely redundant; low overlap means they are complementary, validating the hybrid architecture

**Security:**

- BM25 indices contain the full vocabulary of your corpus
- An inverted index mapping "BJ2019FD7717" → [doc_id_3, doc_id_17] is functionally a lookup of which policy chunks reference that account number
- If your corpus includes any customer-specific data, the index is a sensitive artifact
- Treat it with the same access controls as the source data — not as a public lookup table

**Deployment:**

- For your current 17-page corpus: `rank_bm25` in memory is fine, rebuild on every process start (milliseconds)
- At hundreds of pages: persist the index to disk, rebuild nightly
- At thousands of pages: migrate to Elasticsearch or OpenSearch, which provide inverted index + BM25 natively with distributed search, monitoring, and horizontal scaling

---

### 6. Design Decisions and Trade-offs

**Should `classify_by_keyword_rules()` be replaced with BM25?**

- For classification (Chapter 1), no — BM25 is a retrieval ranking algorithm, not a binary classifier
- The rule engine's binary FD/Non-FD/Multiple-Category output is not something BM25 produces naturally
- For RAG retrieval (this chapter), yes — BM25 replaces binary keyword matching with proper ranked retrieval over the 17-page knowledge base
- The two roles are separate: classification still uses the rule engine as one layer; retrieval uses BM25 as the sparse component of hybrid search

**k1 and b: defaults vs. tuning:**

- Defaults (k1=1.2, b=0.75) work well across many IR benchmarks and are a safe starting point
- Tuning requires labeled query-document relevance pairs — which most projects don't have initially
- Practical approach: start with defaults, collect relevance feedback from production (which retrieved chunks led to good answers), tune when you have enough signal
- For your Hinglish corpus: lower b (0.5–0.6) may work better since your short emails have similar lengths and length normalisation matters less; higher k1 (1.5–2.0) may work better since repetition in policy documents is sometimes genuinely meaningful

**BM25 as a pre-filter vs. parallel with dense retrieval:**

- Pre-filter approach: BM25 retrieves top-1000 candidates cheaply, dense search re-scores only those 1000 — faster overall but BM25 acts as a gate, any chunk with zero vocabulary overlap cannot be recovered by dense retrieval
- Parallel approach: BM25 and dense retrieval run independently, result lists merged via RRF (Topic 4) — more expensive but no chunk is permanently excluded by vocabulary mismatch
- For your 64.4% Hinglish corpus with significant vocabulary mismatch between queries and documents, parallel is the safer choice — vocabulary mismatch is not rare, it is the dominant failure mode

**When to rebuild the BM25 index:**

- Every new document changes df(t) for every term it contains → changes IDF for those terms → slightly changes scores for all other documents containing those terms
- At 17 pages: full rebuild whenever a new document is added (milliseconds, trivial)
- At tens of thousands of pages: full rebuild nightly during low-traffic windows
- At millions of pages: accept approximate IDF (computed at last full rebuild, not updated incrementally), periodic full rebuilds

---

### 7. Alternatives and When to Use Each

**Boolean search (what your rule engine does):**
- Binary keyword presence/absence — correct for hard constraints ("must contain a valid FD reference number")
- No ranking, no IDF, no saturation, no length normalisation
- Use for classification gatekeeping, not for retrieval ranking

**TF-IDF:**
- Better than boolean but has both problems BM25 fixes
- Still widely used for classification tasks (TF-IDF features → logistic regression or SVM)
- Your Chapter 0 and Chapter 2 use TF-IDF this way correctly — for generating feature weights for a classifier, not for retrieval ranking
- For retrieval ranking, BM25 consistently outperforms TF-IDF; use TF-IDF only when you need its simpler implementation or when the task is classification, not ranked retrieval

**BM25 (this topic):**
- Industry default for sparse retrieval — use as the baseline and often the primary sparse signal
- Available in `rank_bm25` (Python, simple), Elasticsearch, OpenSearch, Solr (production scale)
- Best when: exact term matching matters, query and document share vocabulary, corpus is large enough for IDF to be stable

**BM25+ and BM25L (variants):**
- BM25 over-penalises very short documents — a single-sentence FAQ entry gets a lower score than it deserves
- BM25+ adds a lower bound on the term contribution so short documents are not excessively penalised
- BM25L adds a different length correction with similar goals
- Worth knowing by name; `rank_bm25` provides both

**SPLADE and learned sparse retrieval (Topic 3):**
- BM25 operates on the surface vocabulary of the document
- SPLADE uses a transformer to expand each document's sparse representation into a learned vocabulary — generating "latent" terms the document implies but never states
- Beats BM25 significantly on retrieval benchmarks; requires a transformer model and GPU at index time
- The right upgrade once BM25 + dense hybrid is measured to be insufficient

---

### 8. Common Mistakes and Production Failures

- **Not lowercasing before indexing** — BM25 is case-sensitive; "FD" and "fd" are different tokens; always lowercase both documents and queries before tokenizing
- **Not normalising Hinglish variants** — "machurity" and "maturity", "byaj" and "interest" are different tokens; BM25 cannot know they mean the same thing; build a normalisation layer or use query expansion
- **Computing IDF on a small corpus** — your 17-page knowledge base produces unreliable IDF estimates; when the corpus grows significantly, recompute and verify retrieval quality does not silently degrade
- **Thresholding on absolute BM25 scores** — scores shift as corpus size changes; always rank relative to other results in the same query, never apply a fixed score threshold
- **Using BM25 as a blocking gate in front of dense retrieval** — for a Hinglish corpus with significant vocabulary mismatch, any chunk BM25 scores at zero is permanently excluded; run them in parallel instead
- **Not monitoring zero-result queries** — every query that returns zero BM25 scores across your corpus is a vocabulary gap that is silently failing; these should be logged and reviewed
- **Assuming BM25 is a drop-in replacement for semantic search** — BM25 cannot bridge "premature withdrawal" and "exit FD early"; it is a complement to dense retrieval, not a replacement
- **Building BM25 over raw email text for classification** — the keyword groups in `fd_keyword_groups.txt` were built from term-frequency counts on the labeled dataset; replacing them with BM25 over the emails without a labeled relevance signal would lose the supervised signal that made the rule engine effective

---

### 9. Lead-Level Interview Questions

**Basic:**

**Q: What is the difference between TF-IDF and BM25?**

A: Both weight words by frequency in the document (TF) and rarity across the corpus (IDF). BM25 adds two corrections TF-IDF lacks. First, term frequency saturation via the k1 parameter — the contribution of a word plateaus after a number of occurrences, so a document mentioning "maturity" 40 times does not score 40× higher than one mentioning it once. Second, document length normalisation via the b parameter — long documents are penalised and short documents are boosted relative to the corpus average, so verbose policy documents do not automatically beat concise FAQ entries.

**Q: Why would BM25 return a score of zero for a query?**

A: When none of the query terms appear anywhere in the document being scored. BM25 — like all sparse retrieval methods — can only score documents based on term overlap. Zero vocabulary overlap between query and document means zero score regardless of semantic relevance. This is vocabulary mismatch, and it is the primary failure mode BM25 cannot solve — which is why dense retrieval exists alongside it.

**Q: What does the k1 parameter control, and how would you choose a value?**

A: k1 controls how fast term frequency saturates. Higher k1 gives more credit to repeated occurrences before the ceiling is reached; lower k1 approaches binary behaviour. Start with the default (1.2), measure Recall@K on a labeled evaluation set, then search over [0.5, 1.0, 1.2, 1.5, 2.0]. For domains where repetition is semantically meaningful, higher k1 is appropriate. For verbose policy documents where repetition is mostly structural noise, lower k1 is better.

**Intermediate:**

**Q: BM25 scores for the same query change when you add more documents to the corpus. Why, and what are the production implications?**

A: Adding new documents changes df(t) for every term they contain, which changes IDF for those terms, which changes BM25 scores for every existing document containing those terms. Scores are not stable. Production implication: never threshold on absolute BM25 scores — "relevant if score > 5" breaks when corpus size changes. Always rank relative to other results in the same query. Re-evaluate retrieval quality after large corpus additions.

**Q: You have a 17-page knowledge base and a 2,500-email corpus in Hinglish. What are the specific BM25 failure modes you would expect and how would you address them?**

A: Three failure modes. First, vocabulary mismatch — 64.4% Hinglish emails use terms like "byaj" and "machurity" that do not appear in policy chunks using "interest" and "maturity" — address with query expansion or a Hinglish-to-English normalisation layer. Second, small corpus IDF instability — 17 pages produces unreliable IDF; terms appearing in 10 of 17 pages appear "rare" which they are not in the full domain — address by expanding the knowledge base before relying on BM25 IDF estimates. Third, short email length making length normalisation less meaningful — emails average 31 words regardless of class, so b may need tuning lower.

**Advanced:**

**Q: A teammate suggests replacing BM25 entirely with dense retrieval since transformers understand meaning better. How do you respond?**

A: Dense retrieval understands meaning better, but it has specific failure modes where BM25 is more reliable. Exact code matching — a query for "BJ2019FD7717" should retrieve that specific record; if the embedding model has never seen this token pattern during training it may return semantically nearby but incorrect records. Novel product names — "Smart Saver FD" launched this week has no training signal in the embedding model; BM25 finds it by exact term match. The measured evidence from Chapter 3 supports this: cosine similarity between an FD email and a Non-FD email in your corpus is only 0.04 apart (0.4528 vs 0.4135) — dense retrieval on short Hinglish emails has very low discriminating power. The right answer is always both in parallel, with RRF merging the ranked lists.

**Q: BM25 Recall@10 on your evaluation set is 0.61. Before adding a dense retrieval component, what steps would you take?**

A: Diagnose the recall misses first. Sample queries where the correct chunk was not in top 10 and categorise why. Common categories: vocabulary mismatch where query uses Hinglish and document uses English — fix with query expansion or normalisation; same concept different spelling ("machurity" vs "maturity") — fix with synonyms or stemming; genuinely semantic query with no keyword overlap ("is my money safe?") — this is the one BM25 cannot fix and requires dense retrieval. Fix what BM25 can fix first so that when dense retrieval is added you can measure its specific contribution, not conflate it with fixable BM25 weaknesses.

**Scenario-based:**

**Q: Your BM25 retrieval latency on 500K chunks is 8ms. A product requirement says 5,000 QPS. How do you think about this?**

A: At 8ms per query single-threaded, a single process handles roughly 125 QPS. For 5,000 QPS you need ~40 parallel processes, achievable with horizontal scaling. The smarter approach: BM25 is an inverted index lookup, embarrassingly parallelisable — shard the corpus across N machines, each handles the query against its shard, a coordinator merges results. 500K chunks sharded across 10 machines gives 50K per shard, ~1ms per shard, and the coordinator bottleneck becomes the limit. Elasticsearch and OpenSearch implement exactly this architecture natively.

---

### 10. Hidden Concepts and Prerequisites

**Inverted index — what makes BM25 fast in production:**

- BM25 in production does not iterate over every document for every query
- An inverted index maps each vocabulary term to a sorted list of (document_id, term_frequency) pairs
- For a 3-term query, the retrieval engine fetches 3 posting lists and merges them — only visiting documents that contain at least one query term
- `rank_bm25` does not build a true inverted index — it scans all documents (O(n) per query), fine for 17 pages, slow at 500K

**Robertson-Spärck Jones probabilistic model:**

- BM25 derives from the Robertson-Spärck Jones model (1976), which frames retrieval as estimating P(relevance | document, query) using Bayes' theorem
- BM25 is a practical approximation of this probability — which is why its IDF formula looks different from TF-IDF's (it approximates a probability, not just counts)
- Understanding this explains why BM25's IDF uses `(N - df(t) + 0.5) / (df(t) + 0.5)` — it is smoothing a Bayesian probability estimate

**Stop words in BM25 vs TF-IDF:**

- BM25's IDF formula naturally suppresses stop words through their very high df(t) — their IDF is near zero without needing an explicit stop-word list
- Including stop words in the index still wastes space and lookup time in production
- In practice: still remove obvious stop words at index time, but BM25 degrades gracefully even if you forget

**Stemming and its interaction with BM25:**

- BM25 operates on exact token strings — "withdrawal" and "withdrew" are different tokens
- Stemming (reducing words to their root form) can increase recall by merging inflections before BM25 sees them
- For English: Porter or Snowball stemmer is standard
- For Hinglish: English stemmers corrupt Hindi transliterated words — this is a known open problem with no clean off-the-shelf solution; query expansion is safer than stemming for your corpus

**BM25 and your Chapter 0 TF-IDF work:**

- Chapter 0 (`02_Text_Representation.ipynb`) computed TF-IDF with `max_features=20` — a 20-word vocabulary for illustration
- Those real numbers (gaya: 0.750, loan: 0.368, hai: 0.272) demonstrate exactly the IDF suppression and TF weighting that BM25 extends
- BM25 would give "gaya" a higher IDF correction than TF-IDF since it appears in 613 of 2,500 emails — moderately common, not rare

---

### 11. Revision Summary

> `classify_by_keyword_rules()` in Chapter 1 is binary sparse retrieval: word present or absent, no frequency weighting, no corpus statistics, no length awareness. TF-IDF adds corpus statistics (IDF suppresses common words, boosts rare ones) but has two mathematical weaknesses: linear TF growth means verbose documents dominate, and cosine normalisation does not fully correct for document length. BM25 fixes both: the k1 parameter applies a diminishing-returns saturation to TF (the 40th occurrence of "maturity" contributes almost nothing marginal), and the b parameter normalises scoring relative to average document length (long SOPs no longer beat short FAQs purely by size). BM25 is the industry default because it is fast (inverted index, millisecond latency), exact-match reliable (FD reference numbers, Hinglish-specific spellings), and zero per-query API cost. Its one unfixable weakness: zero vocabulary overlap between query and document means zero score regardless of semantic relevance — the vocabulary mismatch problem that Dense Retrieval (Topic 2) and Hybrid Search (Topic 4) exist to solve.

---
